In [10]:
# %%
import os
import pandas as pd
import tempfile
import subprocess
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdDistGeom
from openbabel import openbabel as ob
from meeko import MoleculePreparation

In [2]:
# ================================
# User Config
# ================================
receptor_pdbqt = "../data/external/VEGF.pdbqt"  # Your VEGF receptor in pdbqt format
peptide_csv = "../data/processed/sequences_reviewed.csv"   # CSV with peptide sequences

# Predefined binding site coordinates (replace with your own)
center = [15.23, 35.89, 45.12]
size = [20.0, 20.0, 20.0]

In [24]:
# ================================
# Helper function: robust RDKit PDB loading
# ================================
def load_pdb_with_rdkit(pdb_file):
    mol = Chem.MolFromPDBFile(pdb_file, removeHs=False)
    if mol is None:
        mol = Chem.MolFromPDBFile(pdb_file, removeHs=True)
        if mol is not None:
            mol = Chem.AddHs(mol)
    return mol

# ================================
# Ligand Preparation Functions
# ================================
def prepare_peptide_rdkit(sequence):
    """Generate peptide 3D structure with RDKit."""
    mol = Chem.MolFromSequence(sequence)
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    success = AllChem.EmbedMolecule(mol, params)
    if success != 0:
        raise ValueError("RDKit embedding failed")
    AllChem.MMFFOptimizeMolecule(mol)
    tmp_pdb = tempfile.NamedTemporaryFile(delete=False, suffix=".pdb").name
    Chem.MolToPDBFile(mol, tmp_pdb)
    return tmp_pdb

def prepare_peptide_openbabel_pdbqt(sequence):
    """Generate peptide pdbqt directly with OpenBabel CLI."""
    tmp_pdbqt = tempfile.NamedTemporaryFile(delete=False, suffix=".pdbqt").name
    cmd = [
        'obabel',
        f'-:"{sequence}"',
        '-opdbqt',
        '--gen3d',
        '-O', tmp_pdbqt
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"OpenBabel failed: {result.stderr}")
    return tmp_pdbqt

def prepare_peptide_pdbqt(sequence):
    """Try RDKit 3D generation then Meeko processing; fallback to OpenBabel direct pdbqt."""
    try:
        pdb_file = prepare_peptide_rdkit(sequence)
        mol = load_pdb_with_rdkit(pdb_file)
        if mol is None:
            raise ValueError("RDKit failed to load generated pdb file.")
        preparator = MoleculePreparation()
        preparator.prepare(mol)
        ligand_pdbqt = pdb_file.replace(".pdb", ".pdbqt")
        with open(ligand_pdbqt, "w") as f:
            f.write(preparator.write_pdbqt_string())
        return ligand_pdbqt
    except Exception as e:
        print(f"❌ RDKit or Meeko processing failed for {sequence}: {e} → Using OpenBabel direct pdbqt fallback")
        return prepare_peptide_openbabel_pdbqt(sequence)

In [7]:
# ================================
# Docking Function
# ================================
def run_qvina(receptor, ligand, out_pdbqt, center, size):
    cmd = [
        "qvina2",
        "--receptor", receptor,
        "--ligand", ligand,
        "--center_x", str(center[0]),
        "--center_y", str(center[1]),
        "--center_z", str(center[2]),
        "--size_x", str(size[0]),
        "--size_y", str(size[1]),
        "--size_z", str(size[2]),
        "--exhaustiveness", "8",
        "--out", out_pdbqt
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.stdout

In [25]:
# ================================
# Main docking workflow
# ================================
df = pd.read_csv(peptide_csv)
results = []

for idx, row in df.iterrows():
    seq = row["Sequence"]
    print(f"🔄 Docking peptide {idx}: {seq}")
    try:
        ligand_pdbqt = prepare_peptide_pdbqt(seq)
    except Exception as e:
        print(f"❌ Ligand preparation failed for {seq}: {e}")
        results.append({"Sequence": seq, "Score": None, "Status": "Failed ligand prep"})
        continue

    out_pdbqt = tempfile.NamedTemporaryFile(delete=False, suffix=".pdbqt").name
    log = run_qvina(receptor_pdbqt, ligand_pdbqt, out_pdbqt, center, size)

    # Parse docking score
    score = None
    for line in log.splitlines():
        if "REMARK VINA RESULT" in line:
            try:
                score = float(line.split()[3])
            except Exception:
                score = None
            break

    if score is not None:
        results.append({"Sequence": seq, "Score": score, "Status": "Success"})
        print(f"✅ Score: {score} kcal/mol")
    else:
        results.append({"Sequence": seq, "Score": None, "Status": "Docking failed"})
        print(f"❌ Docking failed for {seq}")


🔄 Docking peptide 0: VAGKVAIRKDVPEISGG
❌ RDKit or Meeko processing failed for VAGKVAIRKDVPEISGG: RDKit embedding failed → Using OpenBabel direct pdbqt fallback
❌ Docking failed for VAGKVAIRKDVPEISGG
🔄 Docking peptide 1: WKTMSPKIVLEDKQALF
❌ RDKit or Meeko processing failed for WKTMSPKIVLEDKQALF: RDKit embedding failed → Using OpenBabel direct pdbqt fallback
❌ Docking failed for WKTMSPKIVLEDKQALF
🔄 Docking peptide 2: WWCLECWCYIVKKKQNG
❌ RDKit or Meeko processing failed for WWCLECWCYIVKKKQNG: RDKit embedding failed → Using OpenBabel direct pdbqt fallback
❌ Docking failed for WWCLECWCYIVKKKQNG
🔄 Docking peptide 3: KKWPKYCFCLVIEVLGS
❌ RDKit or Meeko processing failed for KKWPKYCFCLVIEVLGS: RDKit embedding failed → Using OpenBabel direct pdbqt fallback
❌ Docking failed for KKWPKYCFCLVIEVLGS
🔄 Docking peptide 4: PIGICVYPSRNFVPSKD
❌ RDKit or Meeko processing failed for PIGICVYPSRNFVPSKD: RDKit embedding failed → Using OpenBabel direct pdbqt fallback
❌ Docking failed for PIGICVYPSRNFVPSKD
🔄 Doc

In [ ]:
# Save results
pd.DataFrame(results).to_csv("../data/processed/docking_results.csv", index=False)
print("💾 Results saved to docking_results.csv")